In [9]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pickle
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


In [2]:
BASE_DIR = Path.cwd().parent.parent
DATA_DIR     = BASE_DIR / "data"
FEATURES_DIR = DATA_DIR / "dl_features"
MODELS_DIR   = FEATURES_DIR

print("BASE_DIR      :", BASE_DIR)
print("FEATURES_DIR  :", FEATURES_DIR)

BASE_DIR      : C:\Users\DELL\nlp-fake-news-detector-transformers
FEATURES_DIR  : C:\Users\DELL\nlp-fake-news-detector-transformers\data\dl_features


In [4]:
def load_features():
    print("Loading preprocessed features...")

    X_train = np.load(FEATURES_DIR / "X_train_pad.npy")
    X_val   = np.load(FEATURES_DIR / "X_val_pad.npy")
    X_test  = np.load(FEATURES_DIR / "X_test_pad.npy")

    y_train = np.load(FEATURES_DIR / "y_train.npy")
    y_val   = np.load(FEATURES_DIR / "y_val.npy")
    y_test  = np.load(FEATURES_DIR / "y_test.npy")

    with open(FEATURES_DIR / "meta.pkl", "rb") as f:
        meta = pickle.load(f)

    with open(FEATURES_DIR / "tokenizer.pkl", "rb") as f:
        tokenizer = pickle.load(f)

    print(f"  X_train shape : {X_train.shape}")
    print(f"  X_val   shape : {X_val.shape}")
    print(f"  X_test  shape : {X_test.shape}")
    print(f"  max_len       : {meta['max_len']}")
    print(f"  max_words     : {meta['max_words']}")
    print(f"  Vocab size    : {len(tokenizer.word_index)}")

    return X_train, X_val, X_test, y_train, y_val, y_test, meta, tokenizer

In [5]:
X_train, X_val, X_test, y_train, y_val, y_test, meta, tokenizer = load_features()

max_words = meta['max_words']
max_len   = meta['max_len']

Loading preprocessed features...
  X_train shape : (963423, 20)
  X_val   shape : (222329, 20)
  X_test  shape : (296438, 20)
  max_len       : 20
  max_words     : 20000
  Vocab size    : 295251


In [10]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    GRU,
    Dense,
    Dropout
)

In [11]:
model = Sequential([

    Embedding(
        input_dim=max_words,
        output_dim=128,
        input_length=max_len
    ),

    GRU( 64, dropout=0.2, recurrent_dropout=0.2),

    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

In [12]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(),
        tf.keras.metrics.Recall()
    ]
)

In [13]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
early_stop = EarlyStopping(patience=3, restore_best_weights=True)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[early_stop]
)

Epoch 1/10
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 713s 92ms/step - accuracy: 0.7697 - loss: 0.4752 - precision: 0.7717 - recall: 0.7583 - val_accuracy: 0.7948 - val_loss: 0.4386 - val_precision: 0.7802 - val_recall: 0.8141
Epoch 2/10
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 693s 85ms/step - accuracy: 0.8046 - loss: 0.4266 - precision: 0.8011 - recall: 0.8045 - val_accuracy: 0.7977 - val_loss: 0.4337 - val_precision: 0.7928 - val_recall: 0.7997
Epoch 3/10
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 786s 104ms/step - accuracy: 0.8139 - loss: 0.4088 - precision: 0.8115 - recall: 0.8121 - val_accuracy: 0.7981 - val_loss: 0.4364 - val_precision: 0.7899 - val_recall: 0.8059
Epoch 4/10
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 773s 103ms/step - accuracy: 0.8219 - loss: 0.3925 - precision: 0.8212 - recall: 0.8178 - val_accuracy: 0.7955 - val_loss: 0.4439 - val_precision: 0.7854 - val_recall: 0.8066
Epoch 5/10
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 763s 101ms/step - accuracy: 0.8293 - loss: 0.3775 - precision: 0.8293 - recall: 0.8244 - val_acc

In [16]:
loss, accuracy, precision, recall = model.evaluate( X_test, y_test)

print("\nTest Loss      :", loss)
print("Test Accuracy  :", accuracy)
print("Test Precision :", precision)
print("Test Recall    :", recall)

9264/9264 ━━━━━━━━━━━━━━━━━━━━ 82s 9ms/step - accuracy: 0.7970 - loss: 0.4334 - precision: 0.7912 - recall: 0.8005

Test Loss      : 0.43336251378059387
Test Accuracy  : 0.7969693541526794
Test Precision : 0.7911520004272461
Test Recall    : 0.8005051016807556


In [17]:
y_pred = (model.predict(X_test) > 0.5).astype(int)
f1 = f1_score(y_test, y_pred)

print("Test F1 Score  :", f1)

9264/9264 ━━━━━━━━━━━━━━━━━━━━ 76s 7ms/step
Test F1 Score  : 0.7958010734812141
